# Maia Lite 0.3.0 — Diagnóstico de inferência SFT 200k

**Notebook independente de testes (sem treinamento)**, baseado exatamente na arquitetura, LoRA e formato de prompt do notebook de treinamento anexado.

Compara **base 300k vs. LoRA final**, testa PT/EN/ES e conversas de múltiplos turnos, permite testes livres e exporta resultados em JSON/CSV.

> Execute no Colab com GPU L4/A100. Usa checkpoints próprios/confiáveis do Google Drive. Não modifica os pesos nem os checkpoints.


## 1. Ambiente e caminhos

In [ ]:
import os, json, math, time, random, gc, csv, re
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Selecione um runtime com GPU.'
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
SFT_DIR=ROOT/'training_runs/maia_lite_0_3_0_sft_exp_200k_lora_v1'
ADAPTER_CKPT=SFT_DIR/'adapter_latest.pt'
RESULT_DIR=SFT_DIR/'inference_diagnostics'
for p in (BASE_CKPT,ADAPTER_CKPT):
    assert p.is_file(),f'Arquivo ausente: {p}'
assert TOKENIZER_DIR.is_dir()
print('GPU:',torch.cuda.get_device_name(0))
print('Base:',BASE_CKPT)
print('LoRA:',ADAPTER_CKPT)

## 2. Arquitetura original do treinamento — sem alterações

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## 3. Carregar modelo base e tokenizer

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size
assert tokenizer.eos_token_id is not None
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False) # somente checkpoint próprio/confiável
assert 'model' in checkpoint and int(checkpoint.get('step',-1))==300000
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint;gc.collect()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)


## 4. Estrutura LoRA idêntica ao treinamento

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
trainable=[p for p in model.parameters() if p.requires_grad]
print('LoRA trainable parameters:',sum(p.numel() for p in trainable))
assert len(trainable)==cfg.n_layer*2*2


## 5. Carregar adaptador treinado e verificar integridade

In [ ]:
saved=torch.load(ADAPTER_CKPT,map_location='cpu',weights_only=False) # apenas arquivo próprio/confiável
assert isinstance(saved,dict) and 'adapter' in saved and 'config' in saved
config=saved['config']
assert config['base_step']==300000
assert config['rank']==RANK and config['alpha']==ALPHA
assert config['format']=='Usuário:/Assistente:'
state=saved['adapter']
expected={k for k in model.state_dict() if k.endswith('.A') or k.endswith('.B')}
assert set(state)==expected, f'Chaves LoRA incompatíveis: faltam {len(expected-set(state))}, extras {len(set(state)-expected)}'
with torch.no_grad():
    params=dict(model.named_parameters())
    for name,tensor in state.items():
        assert tuple(params[name].shape)==tuple(tensor.shape),name
        params[name].copy_(tensor.to(params[name].device,dtype=params[name].dtype))
UPDATE=int(saved.get('update',-1))
print('Checkpoint update:',UPDATE,'| exemplos treino:',config.get('train_pairs'))
print('LoRA carregado:',len(state),'tensores')
del saved,state
gc.collect()
model.eval()
for block in model.blocks:
    block.qkv.dropout.p=0.0
    block.proj.dropout.p=0.0
assert model.cfg.block_size==2048


## 6. Geração controlada

O treinamento utilizou o formato **`Usuário: ...\nAssistente:\n`** para todos os idiomas. Aqui preservamos exatamente esse formato. A comparação com o modelo-base usa **os mesmos prompts, sementes e parâmetros**. O modelo não tem KV-cache: a geração é deliberadamente simples, porém mais lenta.


In [ ]:
MAX_NEW_TOKENS=100
TEMPERATURE=0.7
TOP_P=0.9
TOP_K=40
SEED=20261121
STOP_ON_ROLE_MARKER=True

def set_adapter(enabled):
    scale=ALPHA/RANK if enabled else 0.0
    for b in model.blocks:
        b.qkv.scale=scale
        b.proj.scale=scale

def build_prompt(turns):
    lines=[]
    for role,text in turns:
        assert role in ('user','assistant')
        lines.append(('Usuário: ' if role=='user' else 'Assistente: ')+text.strip())
    assert turns and turns[-1][0]=='user'
    return '\n'.join(lines)+'\nAssistente:\n'

def sample_token(logits,generator,temperature,top_k,top_p):
    logits=logits.float()
    if temperature<=0:
        return int(torch.argmax(logits).item())
    logits=logits/temperature
    if top_k is not None and top_k>0:
        k=min(top_k,logits.numel())
        cutoff=torch.topk(logits,k).values[-1]
        logits=logits.masked_fill(logits<cutoff,float('-inf'))
    probs=torch.softmax(logits,dim=-1)
    if top_p is not None and 0<top_p<1:
        p_sorted,idx=torch.sort(probs,descending=True)
        cum=torch.cumsum(p_sorted,dim=-1)
        remove=(cum-p_sorted)>top_p
        p_sorted=p_sorted.masked_fill(remove,0)
        p_sorted=p_sorted/p_sorted.sum()
        chosen=torch.multinomial(p_sorted,1,generator=generator)
        return int(idx[chosen].item())
    return int(torch.multinomial(probs,1,generator=generator).item())

@torch.inference_mode()
def generate(prompt,adapter=True,max_new_tokens=None,temperature=None,seed=SEED):
    set_adapter(adapter)
    model.eval()
    max_new_tokens=MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens
    temperature=TEMPERATURE if temperature is None else temperature
    tokens=tokenizer.encode(prompt,add_special_tokens=False)
    assert tokens, 'Prompt vazio'
    # Não truncar silenciosamente o histórico: conservar o sufixo.
    max_prompt=min(768,model.cfg.block_size-max_new_tokens)
    if len(tokens)>max_prompt:
        tokens=tokens[-max_prompt:]
    generator=torch.Generator(device='cuda')
    generator.manual_seed(seed)
    generated=[]
    start=time.monotonic()
    for _ in range(max_new_tokens):
        x=torch.tensor([tokens+generated],device='cuda',dtype=torch.long)
        logits,_=model(x)
        nxt=sample_token(logits[0,-1],generator,temperature,TOP_K,TOP_P)
        if nxt==tokenizer.eos_token_id:break
        generated.append(nxt)
        if STOP_ON_ROLE_MARKER:
            s=tokenizer.decode(generated,skip_special_tokens=True)
            # Não confundir o prefixo inicial de uma resposta com um marcador de nova fala.
            if re.search(r'\n(?:Usuário|Assistente):',s):
                break
    answer=tokenizer.decode(generated,skip_special_tokens=True)
    if STOP_ON_ROLE_MARKER:
        answer=re.split(r'\n(?:Usuário|Assistente):',answer,maxsplit=1)[0]
    return {'answer':answer.strip(),'generated_tokens':len(generated),
            'prompt_tokens':len(tokens),'seconds':round(time.monotonic()-start,2)}


## 7. Teste rápido — verificar que a geração funciona antes da bateria

In [ ]:
quick='Usuário: Olá! Como você está?\nAssistente:\n'
for enabled in (False,True):
    out=generate(quick,adapter=enabled,max_new_tokens=60,seed=SEED)
    print('\n', 'LoRA final' if enabled else 'Base 300k', '=>',repr(out['answer']),out['seconds'],'s')


## 8. Bateria fixa trilíngue e comparativa

In [ ]:
TESTS=[
 ('pt','saudacao','Olá! Como você está?'),
 ('pt','identidade','Quem é você? Responda em uma frase.'),
 ('pt','fato','Qual é a capital do Brasil?'),
 ('pt','instrucao','Diga apenas a palavra azul.'),
 ('pt','logica','Se Ana é mais velha que Bia, quem é mais nova?'),
 ('pt','incerteza','Qual é a minha profissão?'),
 ('pt','explicacao','Explique em duas frases por que chove.'),
 ('en','greeting','Hello! How are you?'),
 ('en','identity','Who are you? Answer in one sentence.'),
 ('en','fact','What is the capital of France?'),
 ('en','instruction','Reply with only the word blue.'),
 ('en','logic','If Alice is older than Bob, who is younger?'),
 ('en','uncertainty','What is my profession?'),
 ('en','explanation','Explain in two sentences why it rains.'),
 ('es','saludo','¡Hola! ¿Cómo estás?'),
 ('es','identidad','¿Quién eres? Responde en una frase.'),
 ('es','hecho','¿Cuál es la capital de Argentina?'),
 ('es','instruccion','Responde solo con la palabra azul.'),
 ('es','logica','Si Ana es mayor que Beto, ¿quién es menor?'),
 ('es','incertidumbre','¿Cuál es mi profesión?'),
 ('es','explicacion','Explica en dos frases por qué llueve.')
]
results=[]
for i,(lang,category,question) in enumerate(TESTS):
    prompt=build_prompt([('user',question)])
    print(f'\n[{i+1}/{len(TESTS)}] {lang}/{category}: {question}')
    for enabled in (False,True):
        result=generate(prompt,adapter=enabled,seed=SEED+i,max_new_tokens=MAX_NEW_TOKENS)
        item={'lang':lang,'category':category,'question':question,
              'model':'lora_final' if enabled else 'base_300k',**result}
        results.append(item)
        print('  '+item['model']+': '+repr(result['answer']))


## 9. Conversa de múltiplos turnos — teste de memória local

In [ ]:
SCENARIOS={
 'pt':[
    'Meu cachorro se chama Tico. Guarde esse nome durante nossa conversa.',
    'Qual é o nome do meu cachorro?',
    'Agora responda em uma frase o que você sabe sobre ele.'
 ],
 'en':[
    'My cat is named Luna. Remember that name during our conversation.',
    'What is my cat called?',
    'What do you know about my cat?'
 ],
 'es':[
    'Mi perro se llama Paco. Recuerda ese nombre durante esta conversación.',
    '¿Cómo se llama mi perro?',
    '¿Qué sabes de mi perro?'
 ]
}
multi_results=[]
for lang,questions in SCENARIOS.items():
    for enabled in (False,True):
        history=[]
        print('\n',lang,'LoRA' if enabled else 'Base')
        for turn,q in enumerate(questions):
            history.append(('user',q))
            r=generate(build_prompt(history),adapter=enabled,max_new_tokens=90,seed=SEED+turn)
            history.append(('assistant',r['answer']))
            multi_results.append({'lang':lang,'model':'lora_final' if enabled else 'base_300k',
                                  'turn':turn+1,'question':q,**r})
            print('  Usuário:',q)
            print('  Assistente:',repr(r['answer']))


## 10. Teste interativo: escreva sua pergunta e escolha o modelo

In [ ]:
QUESTION='Explique brevemente o que é inteligência artificial.'
USE_LORA=True
TEMPERATURE_INTERACTIVE=0.7 # 0 = greedy, determinístico
MAX_TOKENS_INTERACTIVE=160
answer=generate(build_prompt([('user',QUESTION)]),adapter=USE_LORA,
                temperature=TEMPERATURE_INTERACTIVE,max_new_tokens=MAX_TOKENS_INTERACTIVE)
print('Modelo:', 'LoRA final' if USE_LORA else 'Base 300k')
print('Pergunta:',QUESTION)
print('Resposta:',answer['answer'])
print('Tokens:',answer['generated_tokens'],'| Tempo:',answer['seconds'],'s')


## 11. Exportar resultados e diagnóstico

In [ ]:
RESULT_DIR.mkdir(parents=True,exist_ok=True)
payload={'base_checkpoint':str(BASE_CKPT),'adapter_checkpoint':str(ADAPTER_CKPT),
         'adapter_update':UPDATE,'generation':{'temperature':TEMPERATURE,'top_p':TOP_P,
         'top_k':TOP_K,'max_new_tokens':MAX_NEW_TOKENS,'seed':SEED},
         'single_turn':results,'multi_turn':multi_results}
json_path=RESULT_DIR/'diagnostic_results.json'
json_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
csv_path=RESULT_DIR/'single_turn_results.csv'
with csv_path.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)
print('JSON:',json_path)
print('CSV:',csv_path)
print('Nenhuma métrica automática aqui prova qualidade; compare as respostas e anote erros.')


## 12. Como interpretar

- Se **base e LoRA** falham igualmente, investigue a capacidade do modelo-base e o pré-treinamento.
- Se **base funciona e LoRA piora**, investigue dados de SFT, máscara de loss e hiperparâmetros.
- Se as respostas **mudam drasticamente com temperatura zero**, investigue a estratégia de amostragem.
- Se aparecem `Usuário:` ou `Assistente:` dentro das respostas, investigue os delimitadores e a montagem do corpus.
- Se a resposta é boa no primeiro turno e piora depois, investigue histórico, limites de contexto e distribuição dos exemplos multi-turno.

**Atenção:** a máscara de loss do treino é deslocada corretamente para o próximo token, mas o teste não substitui auditoria da qualidade do corpus. O experimento não deve ser distribuído antes da verificação das licenças.
